In [ ]:
!pip install -q transformers accelerate datasets

In [ ]:
import torch
from collections import Counter
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification

MODEL_NAME = "dslim/bert-base-NER"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForTokenClassification.from_pretrained(MODEL_NAME).to(device)
model.eval()

dataset = load_dataset("lhoestq/conll2003")
label_names = ["O","B-PER","I-PER","B-ORG","I-ORG","B-LOC","I-LOC","B-MISC","I-MISC"]

In [ ]:
def bert_ner_batch(batch_tokens):
    encoded = tokenizer(
        batch_tokens,
        is_split_into_words=True,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512
    )
    word_id_lists = [encoded.word_ids(batch_index=i) for i in range(len(batch_tokens))]
    model_inputs = {k: v.to(device) for k, v in encoded.items()}
    with torch.inference_mode():
        predictions = model(**model_inputs).logits.argmax(dim=-1).cpu().tolist()
    results = []
    for sentence_predictions, word_ids, tokens in zip(predictions, word_id_lists, batch_tokens):
        labels = []
        previous_word_id = None
        for prediction, word_id in zip(sentence_predictions, word_ids):
            if word_id is None or word_id == previous_word_id:
                previous_word_id = word_id
                continue
            labels.append(model.config.id2label[int(prediction)])
            previous_word_id = word_id
        results.append(labels[:len(tokens)])
    return results

def bert_ner(tokens):
    return bert_ner_batch([tokens])[0]

In [ ]:
def get_entity_set(tokens, tags):
    entities = set()
    start = None
    entity_type = None
    for i, tag in enumerate(tags):
        if isinstance(tag, int):
            tag = label_names[tag]
        if tag.startswith("B-"):
            if start is not None:
                entities.add((start, i, entity_type))
            start = i
            entity_type = tag[2:]
        elif tag.startswith("I-"):
            if start is None:
                start = i
                entity_type = tag[2:]
        else:
            if start is not None:
                entities.add((start, i, entity_type))
            start = None
            entity_type = None
    if start is not None:
        entities.add((start, len(tokens), entity_type))
    return entities

entity_frequency = Counter()
for example in dataset["train"]:
    for start, end, entity_type in get_entity_set(example["tokens"], example["ner_tags"]):
        entity_frequency[" ".join(example["tokens"][start:end])] += 1

def get_entity_category(entity_text):
    frequency = entity_frequency.get(entity_text, 0)
    return "COMMON" if frequency >= 5 else "UNCOMMON" if frequency >= 1 else "UNSEEN" 

In [ ]:
batch_size = 16
bert_y_true = []
bert_y_pred = []

test_tokens = dataset["test"]["tokens"]
for start in range(0, len(test_tokens), batch_size):
    batch = test_tokens[start:start + batch_size]
    bert_y_pred.extend(bert_ner_batch(batch))
    bert_y_true.extend([[label_names[x] for x in tags] for tags in dataset["test"]["ner_tags"][start:start + batch_size]])

bert_total_gold = bert_total_predicted = bert_total_correct = 0
bert_category_total = Counter()
bert_category_correct = Counter()

for example, gold_tags, predicted_tags in zip(dataset["test"], bert_y_true, bert_y_pred):
    tokens = example["tokens"]
    gold_entities = get_entity_set(tokens, gold_tags)
    predicted_entities = get_entity_set(tokens, predicted_tags)
    correct_entities = gold_entities & predicted_entities
    bert_total_gold += len(gold_entities)
    bert_total_predicted += len(predicted_entities)
    bert_total_correct += len(correct_entities)
    for start, end, entity_type in gold_entities:
        category = get_entity_category(" ".join(tokens[start:end]))
        bert_category_total[category] += 1
        if (start, end, entity_type) in correct_entities:
            bert_category_correct[category] += 1

bert_precision = bert_total_correct / bert_total_predicted
bert_recall = bert_total_correct / bert_total_gold
bert_f1 = 2 * bert_precision * bert_recall / (bert_precision + bert_recall)

print(f"Gold entities: {bert_total_gold}")
print(f"Predicted entities: {bert_total_predicted}")
print(f"Correct entities: {bert_total_correct}")
print(f"Precision: {bert_precision:.4%}")
print(f"Recall: {bert_recall:.4%}")
print(f"F1: {bert_f1:.4%}")
for category in ["COMMON", "UNCOMMON", "UNSEEN"]:
    total = bert_category_total[category]
    correct = bert_category_correct[category]
    recall = correct / total if total else 0
    print(f"{category}: {correct}/{total} ({recall:.4%})")

In [ ]:
idx = 4
example = dataset["test"][idx]
tokens = example["tokens"]
gold_entities = get_entity_set(tokens, example["ner_tags"])
bert_entities = get_entity_set(tokens, bert_y_pred[idx])
print(" ".join(tokens))
print("Gold:", [(" ".join(tokens[s:e]), t) for s, e, t in sorted(gold_entities)])
print("BERT:", [(" ".join(tokens[s:e]), t) for s, e, t in sorted(bert_entities)])
print("Uzbekistan frequency:", entity_frequency.get("Uzbekistan", 0))
print("Uzbekistan category:", get_entity_category("Uzbekistan"))